# 抓取并归档 Starlink / Amazon Kuiper / OneWeb

保存 UTC 时间命名的原始 TLE 和完整 OMM JSON；不生成虚拟 TLE。六位 NORAD 编号无法放入传统 TLE，后续优先读取 JSON。下载地址、哈希、数量、历元范围和缺失编号写入 snapshot manifest。

首次先安装同目录 `requirements.txt`，详见 `README.md`。
原始 TLE 接口失败而 JSON 成功时，另存明确标注 `derived_from_omm` 的五位编号 TLE 转换文件；六位编号只保留在 OMM 中。


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

# Locate the shared module from either the repository or notebook directory.
ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in [base, base / "generate satellite"]
             if (p / "satellite_pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the WorldNTN repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from satellite_pipeline import (
    download_snapshot, initialize_scenario, read_scenario, load_snapshot,
    export_tracks, preview_export, merge_exports, stamp,
)
# If dependencies are missing, run this in a separate notebook cell:
# %pip install -r "{ROOT / 'requirements.txt'}"
print("Workspace:", ROOT)


## 抓取设置
`auto`：复用两小时内快照；`refresh`：重新抓取；`local`：完全离线。各星座每种格式只请求一次，不循环重试。

In [ ]:
MODE = "auto"
RESET_START_TIME = False  # True starts a new common simulation epoch.
snapshot = download_snapshot(ROOT, mode=MODE)
print(snapshot)


In [ ]:
manifest = json.loads(snapshot.read_text())
rows = []
for constellation, entry in manifest["constellations"].items():
    for fmt in ("json", "tle", "derived_tle"):
        if fmt not in entry:
            continue
        rows.append({"constellation": constellation, "format": fmt, **entry[fmt]})
display(pd.DataFrame(rows))
for constellation, entry in manifest["constellations"].items():
    print(constellation, "OMM IDs absent from TLE:", len(entry["omm_ids_absent_from_tle"]))


## 固定公共时间轴和输入快照
首次将当前 UTC 写入 `scenario.json`；后续不自动改时间。所有真实/虚拟 notebook 读取同一文件。修改观测点、时间、门限后应重跑需要合并的全部 CSV。

默认：原参考坐标 39.95697°N、−105.16033°E、1660 m；24 小时、120 秒间隔、720 个采样点、最低仰角 25°。

In [ ]:
scenario = initialize_scenario(snapshot, ROOT, reset_start=RESET_START_TIME)
display(scenario)


## 已审批轨道资料
`orbit_info/` 与 `tle/` 并列，包含官方 PDF、逐来源记录和机器可读参数表。详细批准范围和模拟假设见 `orbit_info/SOURCES.md`。

In [ ]:
display(pd.read_csv(ROOT / "orbit_info" / "approved_shells.csv"))
